# Experimento — Perceptron multicamadas (MLP)

Rede treinada do zero para prever `Churn`. O estudo avança em **blocos sequenciais**: cada bloco testa uma intervenção sobre o campeão do anterior e só a aceita se superar o ruído, para sabermos o que melhora o modelo e o que não melhora.

1. **Referência e busca:** o MLP padrão, as referências e o Optuna, com KS como critério e as 5 dobras oficiais.
2. **Erros consistentes:** quais clientes várias "lentes" erram juntas e o que isso revela sobre como os modelos veem os dados. Cada padrão vira uma hipótese.
3. **Insights em sequência:** cada hipótese é um bloco, que herda o campeão do anterior.
4. **Registro evolutivo:** o que cada etapa do pré-processamento muda no MLP e no XGBoost.

**Regra de aceite:** ganhar em pelo menos 80% das divisões repetidas do treino e ter ganho médio maior que a régua de ruído (quanto o KS do próprio campeão muda só de trocar a semente).

O teste só é usado no fim, uma vez, com a sequência congelada e `REVELAR_TESTE = True`. Até lá tudo usa apenas o treino.

## Como executar

### No Kaggle (execução principal)

1. **Notebook:** *Code → New Notebook → File → Import Notebook* → `experimentos/mlp/mlp.ipynb`.
2. **Dados:** *Add Input* → dataset `customers-churned-in-telecom-services`. Sem ele, o notebook tenta baixar via `kagglehub`.
3. **Secrets:** *Add-ons → Secrets* → `GITHUB_TOKEN` e `WANDB_API_KEY`, marcados como anexados. Não suba o `.env` para o Kaggle.
4. **Settings:** *Internet* **On**. O MLP é pequeno, então o acelerador é opcional.
5. Rode primeiro com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

Para retomar uma execução: use o mesmo `RUN_NAME` e `RESUME_FROM = "github"`.

### Localmente

Preencha o `.env` da raiz do repositório (é opcional) e rode de qualquer pasta do projeto. O CSV é procurado em `dados/telco_ibm.csv`.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/mlp"             # branch com o código
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "mlp-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
N_TRIALS = 100                   # mesmo N para todos os modelos (pré-proposta)
N_SEMENTES = 5                   # sementes da avaliação final e das lentes de erro
K_CONFIGS = 5                    # melhores configurações do estudo usadas como lentes de erro
REPETICOES = 5                   # divisões de treino repetidas no registro evolutivo e nas hipóteses
REVELAR_TESTE = False            # True só com a configuração congelada: o teste é usado uma única vez
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa trials novos e encerra limpo (o Kaggle corta em 12 h)
BACKUP_MIN = 30                  # envia os resultados ao GitHub a cada N minutos durante a busca

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "mlp" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

sys.path[:0] = [str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp")]
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("mlp-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv()
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "5"
    N_TRIALS, N_SEMENTES, K_CONFIGS, REPETICOES = 3, 2, 2, 2
    print("MODO_TESTE: 3 trials, 5 épocas. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def salvar_json(nome, obj):
    json.dump(obj, open(os.path.join(PASTA, nome), "w"), indent=1)
    salvar()


def tempo_restante():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO)


def continuar():
    return tempo_restante() > 0


import numpy as np, optuna, pandas as pd, torch
from scipy.stats import ks_2samp
import comum, modelo as M, avaliar as A, erros as ER, evolucao as EV, blocos as BL

pd.set_option("display.max_columns", None, "display.width", 200, "display.max_colwidth", 60)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | device: {M.DEVICE} | GitHub: {'sim' if GITHUB_TOKEN and RESULTS_REPO_URL else 'não'}")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="mlp", config=dict(modelo="mlp", n_trials=N_TRIALS, lote=M.LOTE, paciencia=M.PACIENCIA, teste=MODO_TESTE))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def _apos_bloco(r):
    if wandb_run is not None:
        wandb_run.log({"bloco": r["nome"], "regua": r["regua"], "ks_campeao": max(l["ks"] for l in r["tabela"] if l["decisao"] in ("base", "ACEITA"))})
    salvar()


BL.configurar(PASTA, REPETICOES, _apos_bloco)

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
folds, final, colunas = comum.carregar_dobras()
ind_val, _ = comum.indices_dobras()
df_ = comum.carregar_df()
for i, f in enumerate(folds):
    assert np.array_equal(df_.loc[ind_val[i], "Churn"].values, f[3].astype(int)), "índices fora de ordem"
    assert abs(f[1].mean() - 0.5) < 1e-6, "treino da dobra deveria estar balanceado"
print(f"{len(df_)} clientes | {len(colunas)} variáveis | validação por dobra: {[len(f[2]) for f in folds]} | índices conferidos")
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
print("KS conferido contra scipy")

# Bloco 0 — Referências

**Pergunta:** onde partimos? O MLP padrão da pré-proposta (1 camada, 10 unidades), uma regressão logística e um XGBoost nas 5 dobras oficiais.

In [ ]:
from sklearn.linear_model import LogisticRegression
ref = {"mlp_padrao": [comum.ks(f[3], M.fit(M.MLP_PADRAO, f, [f[2]], 42 + i)[0][0]) for i, f in enumerate(folds)],
       "regressao_logistica": [comum.ks(f[3], LogisticRegression(max_iter=3000).fit(f[0], f[1]).predict_proba(f[2])[:, 1]) for f in folds],
       "xgboost": [comum.ks(f[3], ER._xgb(42).fit(f[0], f[1]).predict_proba(f[2])[:, 1]) for f in folds]}
display(pd.DataFrame(ref, index=[f"dobra {i}" for i in range(5)]).T.assign(media=lambda d: d.mean(axis=1), dp=lambda d: d.iloc[:, :5].std(axis=1)).round(4))
camp = BL.campeao_inicial(M.MLP_PADRAO)

# Bloco 1 — Busca de hiperparâmetros no Optuna

**Pergunta:** quanto a busca melhora o MLP padrão?

Nove hiperparâmetros: os 8 da pré-proposta (camadas, unidades, ativação, otimizador, taxa, dropout, L2 e perda) mais a **paciência** {5, 10, 20, 30}. Lote 256 e no máximo 10.000 épocas ficam fixos. O trial 0 é a configuração padrão. O estudo fica em SQLite e é retomável.

A busca escolhe nas dobras oficiais, então o melhor trial tende a estar otimista. Por isso, a comparação com o padrão é refeita em divisões novas do treino.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
cb = fluxo.callback_trial(wandb_run, salvar, BACKUP_MIN)
st = M.estudo(N_TRIALS, timeout=max(tempo_restante(), 1), callbacks=[cb])
salvar()
ok = [t for t in st.trials if t.state.name == "COMPLETE"]
print(f"{len(ok)} completos, {len(st.trials) - len(ok)} podados ou interrompidos | tempo de parede: {st.user_attrs.get('tempo_parede_s', 0) / 60:.1f} min")
print(f"KS trial 0 (padrão) = {st.trials[0].value:.4f} -> melhor trial = {st.best_value:.4f} (trial {st.best_trial.number})")
print("Melhor configuração:", st.best_params)
df = st.trials_dataframe()
df = df[["number", "state", "value"] + [c for c in df.columns if c.startswith("params_")]]
display(df.sort_values("value", ascending=False).head(10))

In [ ]:
# Melhor KS por valor de cada variável categórica, quantos trials cada valor recebeu e importâncias
linhas = [dict(hiperparametro=h, valor=str(t.params[h]), ks=t.value) for t in ok for h in ("ativ", "otim", "camadas", "perda", "paciencia")]
display(pd.DataFrame(linhas).groupby(["hiperparametro", "valor"]).ks.agg(["count", "max", "median"]).round(4))
try:
    print("Importâncias:", {k: round(v, 3) for k, v in optuna.importance.get_param_importances(st).items()})
except Exception as e:
    print("Importâncias indisponíveis:", e)

In [ ]:
if continuar():
    camp, t = BL.rodar("b01_busca", "O melhor trial supera o MLP padrão em divisões novas?", {"melhor do Optuna": dict(hp=st.best_params)}, camp)
    display(t.round(4))

### 📝 Análise — Blocos 0 e 1

- **Quanto a busca ganhou sobre o padrão, em divisões novas?** _…_
- **Quais hiperparâmetros importaram? A paciência importou?** _…_
- **Algum otimizador recebeu poucos trials para ser comparado?** _…_

# Erros consistentes do campeão

Um erro de um modelo pode ser acaso. Um erro que sementes, configurações e famílias diferentes cometem juntas diz algo sobre os dados.

**Lentes:** o campeão em várias sementes, as melhores configurações do estudo, uma regressão logística e um XGBoost, nas mesmas dobras. Cada lente decide no próprio limiar de KS máximo. Só entram previsões de validação.

**Três perguntas, nesta ordem:**

1. **Os erros são compartilhados?** `todas_erram` muito acima de `esperado_se_independentes` indica que sim, e `concordancia` mostra entre quais lentes.
2. **Quais variáveis os modelos usam?** `perfil` compara quem é sempre errado com quem é sempre acertado na mesma classe. Esses grupos estão em lados opostos da fronteira, então as diferenças são grandes por construção.
3. **Existe informação para separar os erros?** `mesmo_lado` compara clientes que os modelos veem do mesmo jeito, com rótulos opostos. Um segmento com `q_bh` baixo aqui é a melhor fonte de hipóteses. Se nada aparecer, o limite pode estar na base.

In [ ]:
def analisar_erros(c, rotulo):
    fs, _ = BL.dobras_de(c)
    outras = [p for p in ER.top_configs(st, K_CONFIGS) if p != c["hp"]][:K_CONFIGS - 1]
    P, y_oof = ER.lentes(fs, [c["hp"]] + outras, N_SEMENTES)
    tab, E = ER.classificar(P, y_oof)
    print(f"{P.shape[1]} lentes em {len(P)} clientes")
    print({k: round(comum.ks(y_oof.values, P[k].values), 4) for k in P})
    display(tab.tipo.value_counts().to_frame("clientes"))
    display(ER.resumo(tab, E).round(4))
    display(ER.concordancia(E).round(2))
    res = dict(perfil_fn=ER.perfil(tab, 1), perfil_fp=ER.perfil(tab, 0), risco=ER.mesmo_lado(tab, "risco"), fiel=ER.mesmo_lado(tab, "fiel"))
    for nome, titulo in (("perfil_fn", "Churners perdidos x churners pegos"), ("perfil_fp", "Fiéis acusados x fiéis reconhecidos"),
                         ("risco", "MESMO LADO, risco: fiéis acusados x churners pegos"), ("fiel", "MESMO LADO, fiel: churners perdidos x fiéis reconhecidos")):
        r = res[nome]
        print(titulo, f"| {int((r.q_bh < .05).sum())} segmentos com q < 0,05" if len(r) else "| sem dados")
        if len(r):
            display(r[r.q_bh < .05].assign(dif=lambda d: d.pct_nos_erros - d.pct_nos_acertos).sort_values("dif", key=abs, ascending=False).head(12).round(4))
    salvar_json(f"mlp_erros_{rotulo}.json", dict(lentes=list(P.columns), resumo=ER.resumo(tab, E).to_dict("records"), tipos=tab.tipo.value_counts().to_dict(),
                                                 indice=tab.index.tolist(), frac_erro=tab.frac_erro.tolist(), tipo=tab.tipo.tolist(),
                                                 **{k: v.to_dict("records") for k, v in res.items()}))


if continuar():
    analisar_erros(camp, "apos_busca")

### 📝 O que os erros mostram

- **Os erros são compartilhados entre famílias ou são do MLP?** _…_
- **O que separa clientes do mesmo lado com rótulos opostos?** _…_
- **Que hipótese isso gera para o próximo bloco?** _…_

# Insights em sequência

Cada bloco testa uma intervenção sobre o campeão do bloco anterior. **Para acrescentar um insight:** coloque a variável em `BL.CANDIDATAS`, copie um bloco abaixo, dê a ele o próximo número e registre de onde veio a ideia.

**Estes blocos são a sequência do MLP.** As dobras oficiais não mudam, e os outros modelos não são afetados. Uma mudança só vale para todos se o grupo decidir.

As variações abaixo são o ponto de partida, vindas da sondagem inicial. Muitos blocos em sequência aumentam a chance de aceitar algo por acaso, e por isso a régua é exigente e o teste fica para o fim.

## b02_oversampling

**Pergunta:** Tirar o oversampling melhora o KS?

In [ ]:
if continuar():
    camp, t = BL.rodar("b02_oversampling", "Tirar o oversampling melhora o KS?", {"sem oversampling": dict(dados=dict(over=False))}, camp)
    display(t.round(4))

## b03_variaveis_criadas

**Pergunta:** As variáveis criadas no pré-processamento ajudam, agora com os dados na escala certa?

In [ ]:
if continuar():
    camp, t = BL.rodar("b03_variaveis_criadas", "As variáveis criadas no pré-processamento ajudam, agora com os dados na escala certa?", {"sem total_addons e PaymentMethod_Automatic": dict(dados=dict(criadas=False))}, camp)
    display(t.round(4))

## b04_cliente_novo

**Pergunta:** Marcar clientes novos (até 6 meses) ajuda?

In [ ]:
if continuar():
    camp, t = BL.rodar("b04_cliente_novo", "Marcar clientes novos (até 6 meses) ajuda?", {"cliente_novo": dict(extras=["cliente_novo"])}, camp)
    display(t.round(4))

## b05_delta_preco

**Pergunta:** A diferença entre o preço atual e a média já paga ajuda?

In [ ]:
if continuar():
    camp, t = BL.rodar("b05_delta_preco", "A diferença entre o preço atual e a média já paga ajuda?", {"delta_preco": dict(extras=["delta_preco"])}, camp)
    display(t.round(4))

## b06_mtm_fibra

**Pergunta:** A interação entre contrato mensal e fibra ajuda?

In [ ]:
if continuar():
    camp, t = BL.rodar("b06_mtm_fibra", "A interação entre contrato mensal e fibra ajuda?", {"mtm_fibra": dict(extras=["mtm_fibra"])}, camp)
    display(t.round(4))

## b07_preco_por_servico

**Pergunta:** O preço por serviço assinado ajuda?

In [ ]:
if continuar():
    camp, t = BL.rodar("b07_preco_por_servico", "O preço por serviço assinado ajuda?", {"preco_por_servico": dict(extras=["preco_por_servico"])}, camp)
    display(t.round(4))

## Evolução do campeão

O que cada bloco testou, o que foi aceito e quanto mudou. É esta tabela que responde o que melhora o modelo e o que não melhora.

In [ ]:
evo = BL.historico()
display(evo.round(4))
print("Campeão atual:", json.dumps(camp, ensure_ascii=False))
salvar_json("mlp_historico_blocos.json", evo.to_dict("records"))

In [ ]:
# Os erros mudaram depois dos insights?
if continuar():
    analisar_erros(camp, "apos_insights")

### 📝 Análise da sequência

- **O que foi aceito e o que foi rejeitado?** _…_
- **Os erros consistentes mudaram?** _…_
- **Algum achado vale levar para o grupo?** _…_

# Registro evolutivo do pré-processamento

O que cada etapa muda, de forma cumulativa, no MLP e no XGBoost. Cada `delta` é contra a etapa anterior, pareado por divisão do treino. Imputação e codificação ficam sempre, porque sem elas os modelos não rodam. O MLP usa a melhor configuração do estudo, e o XGBoost, uma configuração fixa.

In [ ]:
if continuar():
    r_evo = EV.registro(st.best_params, REPETICOES)
    display(r_evo.round(4))
    salvar_json("mlp_evolucao.json", r_evo.to_dict("records"))

### 📝 Análise do registro evolutivo

- **Qual etapa mais mudou o MLP? E o XGBoost?** _…_

# Avaliação final no teste

Só com a sequência congelada, e uma única vez. Saem duas linhas:

- **`mlp.json`:** a melhor configuração do Optuna com os dados oficiais. É a linha **comparável com os outros modelos**.
- **`mlp_campeao.json`:** o campeão da sequência de blocos, com o que foi aceito. Mostra quanto os insights renderam fora da validação.

Nas duas, o modelo é treinado no treino inteiro, em 5 sementes, pela mediana das melhores épocas nas dobras, e o limiar sai da validação.

In [ ]:
if REVELAR_TESTE and continuar():
    A.main(N_SEMENTES)
    r = json.load(open(os.path.join(PASTA, "mlp.json")))
    fin = BL.final_campeao(camp, N_SEMENTES)
    salvar_json("mlp_campeao.json", fin)
    print(f"Oficial (Optuna): KS = {r['teste']['ks']:.4f} | IC 95% {np.round(r['teste']['ks_ic95'], 4)} | {r['melhor_config']}")
    print(f"Campeão dos blocos: KS = {fin['teste']['ks']:.4f} | IC 95% {np.round(fin['teste']['ks_ic95'], 4)} | {camp}")
    display(pd.DataFrame(r["ce_vs_mse"]["teste"]).T[["ks", "auroc", "ce", "mse", "ce_corr", "f1", "precision", "recall"]].round(4))
else:
    print("Teste não revelado (REVELAR_TESTE = False).")

In [ ]:
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(melhor_ks_validacao=st.best_value, **{f"melhor/{k}": v for k, v in st.best_params.items()}))
    wandb_run.finish()
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))